# Lab 22 — API vs RAG vs Tools vs Training Decision Lab

**Goal:** turn a real product requirement into a transparent architecture decision using quality, freshness, latency, privacy, volume, capital, and control constraints.

Free-first: all calculations run on CPU. Prices below are synthetic placeholders for teaching; replace them with dated real quotes for a proposal.


## 1. Start with the failure, not the method

Classify the dominant bottleneck: knowledge access, stable behavior, external action/state, domain/language representation, latency/cost, or privacy/control.

The first intervention should be the smallest one that can change the measured bottleneck.


In [ ]:
alternatives=["API","API+RAG","API+tools","SFT/LoRA","continued pretraining","pretraining"]
criteria=["quality_gap","freshness","privacy","latency","volume","capital","control"]
print("Alternatives:", alternatives)
print("Criteria:", criteria)


## 2. Requirement profile

Scores are user-editable planning inputs, not universal rankings. Give each criterion a 1–5 importance value for the workload.


In [ ]:
profile={"quality_gap":4,"freshness":5,"privacy":3,"latency":4,"volume":4,"capital":1,"control":2}
for k,v in profile.items(): print(f"{k}: {v}")


## 3. Request economics

These prices are synthetic. The goal is to expose the calculation chain, not provide a current vendor quote.


In [ ]:
requests_month=1_000_000
base_input_tokens=900
retrieved_tokens=700
output_tokens=250
input_price_per_million=3.0
output_price_per_million=12.0
retrieval_cost_per_request=0.001
tool_cost_per_request=0.0005

api_monthly=requests_month*(base_input_tokens/1e6*input_price_per_million + output_tokens/1e6*output_price_per_million)
rag_monthly=requests_month*((base_input_tokens+retrieved_tokens)/1e6*input_price_per_million + output_tokens/1e6*output_price_per_million + retrieval_cost_per_request)
tool_monthly=api_monthly + requests_month*tool_cost_per_request

print("API monthly model cost:", round(api_monthly,2))
print("RAG monthly model+retrieval cost:", round(rag_monthly,2))
print("API+tools monthly model+tool cost:", round(tool_monthly,2))


## 4. Break-even for training

If a training campaign costs C and reduces serving cost by s per successful task, a simple break-even estimate is C / s.


In [ ]:
training_cost=1000.0
saving_per_successful_task=0.01
break_even_tasks=training_cost/saving_per_successful_task
print("Break-even successful tasks:", break_even_tasks)


## 5. Update cadence

Retraining is repeated cost. RAG moves more of the update path into indexing and per-request context processing.


In [ ]:
training_lifecycle=5000.0
updates_per_month=[0,1,4,30]
for u in updates_per_month:
    print(f"{u:>2} updates/month -> {training_lifecycle*u:,.0f} training lifecycle cost/month")


## 6. Decision worksheet

Record measured or explicitly assumed values for quality, freshness, p95 latency, one-time cost, recurring cost, privacy/control, regression risk, and operational complexity.


In [ ]:
decision_record={
  "problem":"replace with the actual user problem",
  "baseline":"model/prompt/version",
  "dominant_bottleneck":"knowledge | behavior | action | representation | economics | privacy",
  "candidate":"API | RAG | tools | SFT/LoRA | continued PT | pretraining",
  "quality_metric":"",
  "quality_result":"",
  "p95_latency_ms":"",
  "one_time_cost":"",
  "recurring_monthly_cost":"",
  "regression_risk":"",
  "evidence_url":"",
  "next_gate":""
}
decision_record


## 7. Critical-thinking prompts

1. Which assumption, if wrong, would reverse your architecture choice?
2. Which cost is missing from your initial estimate?
3. Does the proposed method directly change the measured bottleneck?
4. What experiment would falsify the decision most cheaply?

### Expected principle
There is no universal architecture winner. A defensible decision is supported by a controlled baseline, explicit assumptions, workload-specific quality evidence, resource accounting, and a clear next gate.


## Primary sources

- RAG: https://arxiv.org/abs/2005.11401
- Toolformer: https://arxiv.org/abs/2302.04761
- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
- Stanford CS336: https://cs336.stanford.edu/


## 7. Research exercise — decide what the model actually needs


In [ ]:
from datasets import load_dataset
import statistics

ds=load_dataset("CohereLabs/aya_dataset",split="train",streaming=True)
rows=[]
for row in ds:
    p=str(row.get("inputs","")); t=str(row.get("targets",""))
    if p and t: rows.append((p,t))
    if len(rows)>=300: break

for label,fn in [
    ("short_lookup",lambda p: len(p)<120),
    ("long_context",lambda p: len(p)>600),
    ("instruction",lambda p: any(x in p.lower() for x in ["write","explain","translate","summarize"])),
]:
    subset=[(p,t) for p,t in rows if fn(p)]
    print(label,"count=",len(subset),"mean_prompt_chars=",round(statistics.mean([len(p) for p,t in subset]),1) if subset else 0))

print("\nDecision test: API = external model capability; RAG = external knowledge/context; tool = external action/computation; training = persistent model-behavior change.")


<details><summary><strong>Expected findings / answer</strong></summary>

The same user-visible request can map to different architectures. If the missing ingredient is current/private knowledge, RAG may be appropriate; if it is an external action or deterministic computation, a tool may be better; if the desired behavior is stable and repeated, training can be justified. An API is a model-access mechanism, not automatically a solution to a knowledge or behavior gap. The exercise is a **problem decomposition decision**, not a leaderboard.
</details>
